In [1]:
from dataclasses import dataclass, asdict

@dataclass
class TrainingConfig:
    # ── Model ──────────────────────────────────────────────
    model_name: str      = "t5-base"       # or "t5-large", "google/flan-t5-base"
    task_prefix: str     = "fix recipe: "  # prepended to every corrupted input

    # ── Tokenisation ───────────────────────────────────────
    max_input_length: int  = 64
    max_target_length: int = 64

    # ── Training ───────────────────────────────────────────
    learning_rate: float   = 3e-4
    weight_decay: float    = 0.01
    num_train_epochs: int  = 3
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size: int  = 8
    gradient_accumulation_steps: int = 4   # effective batch = 8 × 4 = 32
    warmup_ratio: float    = 0.06          # fraction of total steps used for warmup
    max_grad_norm: float   = 1.0

    # ── Checkpointing ──────────────────────────────────────
    save_steps: int        = 500
    eval_steps: int        = 500
    logging_steps: int     = 50
    load_best_model_at_end: bool = True
    metric_for_best_model: str   = "eval_rougeL"
    greater_is_better: bool      = True

    # ── Reproducibility ────────────────────────────────────
    seed: int = 42

cfg = TrainingConfig()
print(asdict(cfg))

{'model_name': 't5-base', 'task_prefix': 'fix recipe: ', 'max_input_length': 64, 'max_target_length': 64, 'learning_rate': 0.0003, 'weight_decay': 0.01, 'num_train_epochs': 3, 'per_device_train_batch_size': 8, 'per_device_eval_batch_size': 8, 'gradient_accumulation_steps': 4, 'warmup_ratio': 0.06, 'max_grad_norm': 1.0, 'save_steps': 500, 'eval_steps': 500, 'logging_steps': 50, 'load_best_model_at_end': True, 'metric_for_best_model': 'eval_rougeL', 'greater_is_better': True, 'seed': 42}


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

# ── Quick sanity check ──────────────────────────────────────────────────────
sample_corrupted = "Ingredientss: 2 cup flur, 1 eg, salt. Diretions: mix al togeter."
sample_clean     = "Ingredients: 2 cups flour, 1 egg, salt. Directions: mix all together."

enc = tokenizer(
    cfg.task_prefix + sample_corrupted,
    max_length=cfg.max_input_length,
    truncation=True,
    return_tensors="pt"
)
dec = tokenizer(sample_clean, max_length=cfg.max_target_length, truncation=True)

print(f"Tokenizer:        {cfg.model_name}")
print(f"Vocab size:       {tokenizer.vocab_size:,}")
print(f"Input token len:  {enc['input_ids'].shape[1]}")
print(f"Target token len: {len(dec['input_ids'])}")

/app/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Tokenizer:        t5-base
Vocab size:       32,100
Input token len:  30
Target token len: 20


In [3]:
import random
import torch
from torch.utils.data import Dataset

random.seed(cfg.seed)

# ── Minimal synthetic examples ──────────────────────────────────────────────
CORRUPTIONS = [
    ("Ingredientss: 2 cup flur, 1 eg, salt tt taste.",
     "Ingredients: 2 cups flour, 1 egg, salt to taste."),
    ("Directons: Pre-heat ovn to 180C. Bake fr 30 min.",
     "Directions: Preheat oven to 180°C. Bake for 30 minutes."),
    ("Title Choclate Cake\nIngredents sugar buttr eggs flowur",
     "Title: Chocolate Cake\nIngredients: sugar, butter, eggs, flour"),
    ("Sevings: 4\nPrep tme 15 mins\nCok time: 30minut",
     "Servings: 4\nPrep time: 15 mins\nCook time: 30 minutes"),
    ("1/2 cupp milk or creme\n2 tsp vanil extrat",
     "1/2 cup milk or cream\n2 tsp vanilla extract"),
]

def make_split(pairs, n):
    """Repeat and shuffle pairs to reach n examples."""
    pool = pairs * (n // len(pairs) + 1)
    random.shuffle(pool)
    return pool[:n]

class MockRecipeDataset(Dataset):
    def __init__(self, pairs, tokenizer, cfg):
        self.tokenizer = tokenizer
        self.cfg       = cfg
        self.inputs    = [cfg.task_prefix + c for c, _ in pairs]
        self.targets   = [t for _, t in pairs]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        model_inputs = self.tokenizer(
            self.inputs[idx],
            max_length=self.cfg.max_input_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        labels = self.tokenizer(
            self.targets[idx],
            max_length=self.cfg.max_target_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids

        # Replace padding token id with -100 so it's ignored in the loss
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      model_inputs.input_ids.squeeze(0),
            "attention_mask": model_inputs.attention_mask.squeeze(0),
            "labels":         labels.squeeze(0),
        }

mock_train_dataset = MockRecipeDataset(make_split(CORRUPTIONS, 200), tokenizer, cfg)
mock_val_dataset   = MockRecipeDataset(make_split(CORRUPTIONS,  40), tokenizer, cfg)

print(f"Train examples: {len(mock_train_dataset)}")
print(f"Val examples:   {len(mock_val_dataset)}")
print(f"\nSample input_ids shape:  {mock_train_dataset[0]['input_ids'].shape}")
print(f"Sample labels shape:     {mock_train_dataset[0]['labels'].shape}")

Train examples: 200
Val examples:   40

Sample input_ids shape:  torch.Size([64])
Sample labels shape:     torch.Size([64])


In [4]:
from transformers import AutoModelForSeq2SeqLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_name)
model = model.to(device)

# ── Parameter budget ────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel:             {cfg.model_name}")
print(f"Total params:      {total_params:,}")
print(f"Trainable params:  {trainable_params:,}")
print(f"Non-trainable:     {total_params - trainable_params:,}")

Device: cpu


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]


Model:             t5-base
Total params:      222,903,552
Trainable params:  222,903,552
Non-trainable:     0


In [5]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader

# ── DataLoaders ─────────────────────────────────────────────────────────────
train_loader = DataLoader(
    mock_train_dataset,
    batch_size=cfg.per_device_train_batch_size,
    shuffle=True,
    num_workers=0,      # set to 2-4 with real data
    pin_memory=device.type == "cuda",
)
val_loader = DataLoader(
    mock_val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

# ── Optimizer ───────────────────────────────────────────────────────────────
# Separate weight decay: don't apply it to biases or LayerNorm params
no_decay = {"bias", "LayerNorm.weight"}
param_groups = [
    {
        "params": [p for n, p in model.named_parameters()
                   if not any(nd in n for nd in no_decay)],
        "weight_decay": cfg.weight_decay,
    },
    {
        "params": [p for n, p in model.named_parameters()
                   if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]
optimizer = AdamW(param_groups, lr=cfg.learning_rate)

# ── LR Scheduler ────────────────────────────────────────────────────────────
steps_per_epoch   = len(train_loader) // cfg.gradient_accumulation_steps
total_steps       = steps_per_epoch * cfg.num_train_epochs
warmup_steps      = int(total_steps * cfg.warmup_ratio)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Steps per epoch:   {steps_per_epoch}")
print(f"Total opt. steps:  {total_steps}")
print(f"Warmup steps:      {warmup_steps}")
print(f"Peak LR:           {cfg.learning_rate}")

Steps per epoch:   6
Total opt. steps:  18
Warmup steps:      1
Peak LR:           0.0003


In [6]:
import mlflow
mlflow.end_run()
EXPERIMENT_NAME     = "recipe-correction-t5"

mlflow.set_tracking_uri("http://172.17.0.1:5000")
mlflow.set_experiment(EXPERIMENT_NAME)

run = mlflow.start_run(run_name=f"{cfg.model_name}__lr{cfg.learning_rate}__ep{cfg.num_train_epochs}")
run_id = run.info.run_id
print(f"MLflow run started: {run_id}")

# ── Log all hyperparameters ──────────────────────────────────────────────────
mlflow.log_params(asdict(cfg))

# ── Log derived schedule values ──────────────────────────────────────────────
mlflow.log_params({
    "effective_batch_size": cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps,
    "total_optimizer_steps": total_steps,
    "warmup_steps": warmup_steps,
    "num_train_examples": len(mock_train_dataset),
    "num_val_examples": len(mock_val_dataset),
    "device": str(device),
    "trainable_params": trainable_params,
})

# ── Log model graph as a text artifact (optional but useful) ─────────────────
import tempfile, os
with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
    f.write(str(model))
    tmp_path = f.name
mlflow.log_artifact(tmp_path, artifact_path="model_summary")
os.unlink(tmp_path)

print("\nLogged to MLflow:")
print(f"  Experiment : {EXPERIMENT_NAME}")
print(f"  Run ID     : {run_id}")
#print(f"  Tracking   : {MLFLOW_TRACKING_URI}/#/experiments/")
print("\nModel setup complete — ready to train.")

# NOTE: do NOT call mlflow.end_run() here.
# The training loop (section 4) will continue logging into this same run
# and close it at the end with mlflow.end_run() or a context manager.

MLflow run started: 9ad14d83933340b991b5f3f677924fce

Logged to MLflow:
  Experiment : recipe-correction-t5
  Run ID     : 9ad14d83933340b991b5f3f677924fce

Model setup complete — ready to train.


In [7]:
!uv pip install evaluate

Using Python 3.11.15 environment at: /app/.venv
Checked 1 package in 207ms


In [ ]:
import torch
import numpy as np
from evaluate import load as load_metric
from itertools import islice

# Ensure rouge is loaded
rouge = load_metric("rouge")

def decode_batch(token_ids: torch.Tensor) -> list[str]:
    """Convert a batch of token id tensors to strings, skipping special tokens."""
    return tokenizer.batch_decode(token_ids, skip_special_tokens=True)

@torch.no_grad()
def evaluate_epoch(model, loader, device, cfg) -> dict:
    """
    Run one full pass over `loader`.
    Returns a dict of scalar metrics ready for mlflow.log_metrics().
    """
    model.eval()
    total_loss = 0.0

    # 1. We iterate over the loader and add batches directly to ROUGE 
    # to avoid blowing up system RAM with a massive list of strings.
    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        total_loss += outputs.loss.item()

        # Greedy decode for ROUGE (cheap; swap for beam search in final eval)
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=cfg.max_target_length,
        )

        # Replace -100 (loss-mask) back to pad_token_id before decoding
        labels_for_decode = labels.clone()
        labels_for_decode[labels_for_decode == -100] = tokenizer.pad_token_id

        preds = decode_batch(generated)
        targets = decode_batch(labels_for_decode)

        # Stream predictions directly to the metric buffer
        rouge.add_batch(predictions=preds, references=targets)
        
        # 2. Free up heavy tensors from VRAM manually
        del generated, outputs, labels_for_decode

    try:
        num_batches = len(loader)
    except TypeError:
        # If it's an islice, we don't have len(), so we count manually or 
        # just use a default for the smoke test
        num_batches = 3 

    avg_loss = total_loss / num_batches

    # 3. Compute everything from the accumulated buffer
    rouge_scores = rouge.compute(use_stemmer=True)

    return {
        "eval_loss":    avg_loss,
        "eval_rouge1":  rouge_scores["rouge1"],
        "eval_rouge2":  rouge_scores["rouge2"],
        "eval_rougeL":  rouge_scores["rougeL"],
    }


# ── Smoke-test the eval function before training ─────────────────────────────
print("Smoke-testing evaluate_epoch() on a small slice of val set...")

# 4. We use islice here so we only evaluate 3 batches for the smoke test.
# This ensures we don't crash before the actual training loop begins!
val_smoke_loader = islice(val_loader, 3)

smoke = evaluate_epoch(model, val_smoke_loader, device, cfg)
for k, v in smoke.items():
    print(f"  {k}: {v:.4f}")
print("OK — eval utility is working.\n")

Smoke-testing evaluate_epoch() on a small slice of val set...


In [8]:
import torch
import numpy as np
import os
import psutil
from evaluate import load as load_metric
from itertools import islice

# Load metric
rouge = load_metric("rouge")

def get_ram_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024**2  # MB

@torch.no_grad()
def evaluate_epoch(model, loader, device, cfg):
    model.eval()
    total_loss = 0.0
    
    print(f"Starting eval. Initial RAM: {get_ram_usage():.2f}MB")
    
    for i, batch in enumerate(loader):
        print(f"--- Batch {i} ---")
        
        # Move to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Loss calculation
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += outputs.loss.item()
        
        # Generation - REDUCE max_new_tokens if it still crashes
        print(f"  Generating... (RAM: {get_ram_usage():.2f}MB)")
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=20, # Hardcoded small for test
        )

        # Decode & Add
        labels_for_decode = labels.clone()
        labels_for_decode[labels_for_decode == -100] = tokenizer.pad_token_id
        
        rouge.add_batch(
            predictions=tokenizer.batch_decode(generated, skip_special_tokens=True),
            references=tokenizer.batch_decode(labels_for_decode, skip_special_tokens=True)
        )

        # AGGRESSIVE CLEANUP
        del outputs, generated, batch, input_ids, attention_mask, labels, labels_for_decode
        torch.cuda.empty_cache()
        
    print(f"Finalizing scores... (RAM: {get_ram_usage():.2f}MB)")
    return rouge.compute(use_stemmer=True)

# Test on ONE SINGLE BATCH
print("Testing on ONE batch...")
try:
    # We use a manual next(iter()) to be absolutely sure we don't loop
    single_batch = [next(iter(val_loader))]
    result = evaluate_epoch(model, single_batch, device, cfg)
    print("Success!", result)
except Exception as e:
    print("Caught error:", e)

Testing on ONE batch...
Starting eval. Initial RAM: 585.60MB
--- Batch 0 ---
  Generating... (RAM: 803.20MB)
Finalizing scores... (RAM: 801.21MB)
Success! {'rouge1': np.float64(0.5454545454545454), 'rouge2': np.float64(0.3), 'rougeL': np.float64(0.5454545454545454), 'rougeLsum': np.float64(0.5454545454545454)}


In [ ]:
import os
import shutil

CHECKPOINT_DIR = "../checkpoints/recipe-correction-tests"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_metric   = -float("inf")   # higher rougeL = better
best_ckpt_dir = None

def save_checkpoint(model, tokenizer, epoch: int, metrics: dict) -> str:
    """
    Saves model + tokenizer to a versioned directory.
    Returns the path so it can be logged to MLflow.
    """
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"epoch-{epoch:02d}")
    os.makedirs(ckpt_path, exist_ok=True)
    model.save_pretrained(ckpt_path)
    tokenizer.save_pretrained(ckpt_path)

    # Write a small sidecar with the metrics for this checkpoint
    meta_path = os.path.join(ckpt_path, "metrics.json")
    import json
    with open(meta_path, "w") as f:
        json.dump(metrics, f, indent=2)

    return ckpt_path

In [ ]:
import time
import math
from torch.cuda.amp import GradScaler, autocast

scaler          = GradScaler(enabled=(device.type == "cuda"))
global_step     = 0
train_loss_accum = 0.0   # accumulates raw loss across gradient_accumulation_steps

print(f"Starting training — {cfg.num_train_epochs} epoch(s), "
      f"{total_steps} optimiser steps total\n")

for epoch in range(1, cfg.num_train_epochs + 1):
    model.train()
    epoch_start = time.time()

    for local_step, batch in enumerate(train_loader, start=1):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        # ── Forward ────────────────────────────────────────────────────────
        with autocast(enabled=(device.type == "cuda")):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            # Scale loss by accumulation steps so the effective gradient
            # matches a single large-batch update
            loss = outputs.loss / cfg.gradient_accumulation_steps

        # ── Backward ───────────────────────────────────────────────────────
        scaler.scale(loss).backward()
        train_loss_accum += loss.item()

        # ── Optimiser step (every `gradient_accumulation_steps` batches) ───
        if local_step % cfg.gradient_accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

            global_step += 1

            # ── Per-step MLflow logging ─────────────────────────────────
            if global_step % cfg.logging_steps == 0:
                avg_train_loss = train_loss_accum / cfg.logging_steps
                current_lr     = scheduler.get_last_lr()[0]

                mlflow.log_metrics(
                    {
                        "train_loss": avg_train_loss,
                        "learning_rate": current_lr,
                        "grad_scaler_scale": scaler.get_scale(),
                    },
                    step=global_step,
                )

                print(
                    f"  Epoch {epoch}/{cfg.num_train_epochs} "
                    f"| step {global_step:>5} "
                    f"| train_loss {avg_train_loss:.4f} "
                    f"| lr {current_lr:.2e}"
                )
                train_loss_accum = 0.0  # reset accumulator after logging

    # ── End-of-epoch validation ─────────────────────────────────────────────
    epoch_elapsed = time.time() - epoch_start
    eval_metrics  = evaluate_epoch(model, val_loader, device, cfg)

    # Add epoch-level training metadata
    eval_metrics["epoch"]          = epoch
    eval_metrics["epoch_time_sec"] = round(epoch_elapsed, 2)
    eval_metrics["samples_per_sec"] = round(
        len(mock_train_dataset) / epoch_elapsed, 1
    )

    mlflow.log_metrics(eval_metrics, step=global_step)

    print(
        f"\nEpoch {epoch} complete ({epoch_elapsed:.0f}s) — "
        + " | ".join(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}"
                     for k, v in eval_metrics.items())
        + "\n"
    )

    # ── Checkpoint: save every epoch, track best ────────────────────────────
    ckpt_path = save_checkpoint(model, tokenizer, epoch, eval_metrics)
    mlflow.log_artifacts(ckpt_path, artifact_path=f"checkpoints/epoch-{epoch:02d}")

    current_metric = eval_metrics[cfg.metric_for_best_model]
    if current_metric > best_metric:
        best_metric   = current_metric
        best_ckpt_dir = ckpt_path
        mlflow.log_metric("best_rougeL", best_metric, step=global_step)
        mlflow.set_tag("best_checkpoint", ckpt_path)
        print(f"  ★ New best {cfg.metric_for_best_model}: {best_metric:.4f} → {ckpt_path}\n")

print(f"Training complete. Best {cfg.metric_for_best_model}: {best_metric:.4f}")
print(f"Best checkpoint: {best_ckpt_dir}")

In [ ]:
import mlflow.pytorch

# ── Log the best checkpoint as the canonical model artifact ─────────────────
if best_ckpt_dir:
    print(f"Logging best checkpoint to MLflow model registry...")

    # Load best weights back in before registering
    from transformers import AutoModelForSeq2SeqLM
    best_model = AutoModelForSeq2SeqLM.from_pretrained(best_ckpt_dir)

    mlflow.pytorch.log_model(
        pytorch_model=best_model,
        artifact_path="best-model",
        registered_model_name="recipe-correction-t5",
    )
    print(f"  Registered as: recipe-correction-t5")

# ── Final summary tags ────────────────────────────────────────────────────
mlflow.set_tags({
    "model_name":        cfg.model_name,
    "task":              "recipe-correction",
    "data_source":       "mock-synthetic",   # ← update when real data lands
    "status":            "complete",
    "best_checkpoint":   best_ckpt_dir or "none",
    f"best_{cfg.metric_for_best_model}": f"{best_metric:.4f}",
})

# ── Close the run opened in §3.6 ─────────────────────────────────────────────
mlflow.end_run()
print(f"\nMLflow run {run_id} closed.")
print(f"View at: {MLFLOW_TRACKING_URI}/#/experiments/")

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load best checkpoint (or replace path with any checkpoint dir)
inf_model     = AutoModelForSeq2SeqLM.from_pretrained(best_ckpt_dir).to(device)
inf_tokenizer = AutoTokenizer.from_pretrained(best_ckpt_dir)
inf_model.eval()

TEST_INPUTS = [
    "Ingredientss: 2 cup flur, 1 eg, salt tt taste.",
    "Directons: Pre-heat ovn to 180C. Bake fr 30 min.",
    "Title Choclate Cake\nIngredents sugar buttr eggs flowur",
]

print("─" * 60)
for raw in TEST_INPUTS:
    prompt = cfg.task_prefix + raw
    enc    = inf_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=cfg.max_input_length,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        out = inf_model.generate(
            **enc,
            max_new_tokens=cfg.max_target_length,
            num_beams=4,
            early_stopping=True,
        )

    decoded = inf_tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"IN  : {raw}")
    print(f"OUT : {decoded}")
    print("─" * 60)